# GazeDecoder V3 — `b_in` Channel Ablation Study



Systematically finds the optimal behavioral-channel composition for the

`b_in` stream of `ChronosXOfficial` via **participant-level 5-fold cross-validation**.



**V3 key changes vs V2:**

- **40 epochs**; best-checkpoint by val F1 (no early stopping)

- **Linear warmup (15%) + cosine annealing** — fast early convergence, stable fine-tuning

- Fixed global seed (42) + per-fold seed for full reproducibility

- Feature layout upgraded to **786d**: Spatial(2) + Semantic(768) + Layer1(8) + Layer2(8)

- All logic lives in `shared/` — this notebook is orchestration-only



**Feature Layout (786d):**

```

x[:, :,   0:2  ]  Spatial  : normalised gaze (x, y)

x[:, :,   2:770]  Semantic : Text(384) + Code(384) embeddings  → ctx stream

x[:, :, 770:778]  Layer1   : 8d per-timestep micro-window behavioral stats

x[:, :, 778:786]  Layer2   : 8d window-level macro stats (broadcast per window)

```



**Sections**

1. Environment Setup

2. Package Installation

3. Dataset

4. Dataset Visualization

5. ChronosX `b_in` Variants Overview

6. 5-fold CV Ablation

7. Results Visualization

8. Statistical Tests

9. Best-Variant Summary


## §1  Environment Setup

In [ ]:
import os
import sys
from pathlib import Path

# Ensure shared/ is importable
HERE = Path.cwd()
if (HERE / "shared").exists():
    sys.path.insert(0, str(HERE))

from shared.config import (
    WINDOW_SIZE,
    STRIDE,
    HIT_THRESHOLD,
    GAZE_DIR,
    ECONTEXT_PATH,
    DS_CACHE_DIR,
    ABL_DIR,
    print_config,
    ensure_dirs,
)
from shared.dataset import build_dataset, get_participant_5fold_splits, FeatureMaskedDataset
from shared.training import run_cv, run_all_models

ensure_dirs()
print_config()
print(f"WINDOW_SIZE={WINDOW_SIZE}, STRIDE={STRIDE}, HIT_THRESHOLD={HIT_THRESHOLD}")
print(f"GAZE_DIR={GAZE_DIR}")
print(f"ECONTEXT_PATH={ECONTEXT_PATH}")
print(f"DS_CACHE_DIR={DS_CACHE_DIR}")
print(f"ABL_DIR={ABL_DIR}")


## §2  Package Installation

In [ ]:
if COLAB:
    %pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
    %pip install -q scikit-learn xgboost lightgbm sentence-transformers tqdm seaborn scipy
    print("✅ Packages installed.")

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from shared.config import (
    SEED,
    GAZE_DIR,
    ECONTEXT_PATH,
    ABL_DIR,
    DS_CACHE_DIR,
    WINDOW_SIZE,
    STRIDE,
    HIT_THRESHOLD,
    FEAT_DIM,
    BEHAV_SLICE,
    set_global_seed,
)
from shared.dataset import build_dataset, get_participant_5fold_splits, FeatureMaskedDataset
from shared.models import CHRONOSX_VARIANTS
from shared.training import run_cv, run_all_models
from shared.viz import (
    results_to_df,
    plot_f1_leaderboard,
    plot_prf_grouped,
    plot_mean_confmat,
    plot_training_curves,
    significance_table,
    scott_knott_esd,
    plot_scott_knott,
)

set_global_seed()
print(f"SEED={SEED} | torch={torch.__version__} | device={'cuda' if torch.cuda.is_available() else 'cpu'}")


## §3  Dataset

In [ ]:
# ── Cache invalidation — run once when feature layout / epochs change ─────────
# Set DRY_RUN=True to preview, then False to actually delete.
import glob, os, pathlib, shutil

DRY_RUN = True   # ← change to False to actually delete

# 1. Old dataset pkl caches (missing _dXXX suffix)
old_pkls = [
    p for p in glob.glob(str(DS_CACHE_DIR / "eyeseq_*.pkl"))
    if "_d" not in pathlib.Path(p).stem
]

# 2. All ablation fold cache dirs (includes Bchan_Spatial / Bchan_L1 run with 20 epochs)
abl_fold_dirs = [d for d in ABL_DIR.iterdir() if d.is_dir()] if ABL_DIR.exists() else []

print(f"{'[DRY RUN] ' if DRY_RUN else ''}Files / dirs to delete:")
for p in old_pkls:
    print(f"  🗑  dataset pkl : {p}")
for d in abl_fold_dirs:
    n_jsons = len(list(d.glob("*.json")))
    print(f"  🗑  ablation dir: {d.name}/  ({n_jsons} json files)")

if not old_pkls and not abl_fold_dirs:
    print("  (nothing to delete — already clean)")

if not DRY_RUN:
    for p in old_pkls:
        os.remove(p)
    for d in abl_fold_dirs:
        shutil.rmtree(d)
    print("\n✅ Done. Re-run the build_dataset cell to rebuild.")
else:
    print(f"\n⚠️  DRY_RUN=True — nothing deleted. Set DRY_RUN=False to confirm.")


In [ ]:
# ── 5-fold CV split sizes (participant-level) ───────────────────────────────
splits = get_participant_5fold_splits(ds, n_folds=5, seed=42, val_policy="rotate")
print("\n5-fold CV split sizes:")
for sp in splits:
    print(
        f"  Fold {sp['fold']}: "
        f"train={len(sp['train_idx'])} "
        f"val={len(sp['val_idx'])} "
        f"test={len(sp['test_idx'])} "
        f"| val_pid={sp['val_pid']} test_pids={sp['test_pids']}"
    )


## §4  Dataset Visualization

In [ ]:
# ── Class distribution ───────────────────────────────────────────────────────
labels_all = np.array([int(s["y"]) for s in ds.samples])
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Bar chart: overall class distribution
axes[0].bar(["Non-issue (0)", "Issue (1)"],
            np.bincount(labels_all), color=["#5B9BD5", "#ED7D31"])
axes[0].set_title("Overall class distribution")
axes[0].set_ylabel("# windows")

# Per-subject distribution (access p_id directly from ds.samples)
pid_all = np.array([s["p_id"] for s in ds.samples])
pids    = sorted(np.unique(pid_all))
issue_frac = [
    np.mean(labels_all[pid_all == p]) for p in pids
]
axes[1].bar(pids, issue_frac, color="#70AD47")
axes[1].axhline(np.mean(labels_all), color="red", linestyle="--",
                label=f"Overall={np.mean(labels_all):.2f}")
axes[1].set_title("Issue-class fraction per subject")
axes[1].set_ylabel("Issue fraction")
axes[1].set_xlabel("Subject ID")
axes[1].tick_params(axis="x", rotation=45)
axes[1].legend()

plt.tight_layout()
plt.show()

# ── 5-fold CV split sizes ────────────────────────────────────────────────────
print("\n5-fold CV split sizes:")
for sp in splits:
    print(
        f"  Fold {sp['fold']}: "
        f"train={len(sp['train_idx'])} "
        f"val={len(sp['val_idx'])} "
        f"test={len(sp['test_idx'])} "
        f"| val_pid={sp['val_pid']} test_pids={sp['test_pids']}"
    )

## §5  ChronosXOfficial — `b_in` Channel Ablation 变体一览

所有变体均基于 `ChronosXOfficial` 骨架（**IIB → Transformer-CLS(3L) → OIB → MLP**固定不变），
仅改变输入到 `b_in` 流的特征通道组合，以及 OIB 的条件向量来源。

### Group A：OIB 条件 = `mean(ctx)` 仅用语义上下文

| 变体名 | `b_in` 内容（dim） | OIB 条件 | 设计意图 |
|---|---|---|---|
| `Bchan_Spatial` | Spatial(2) | mean(ctx) | 纯空间基线 |
| `Bchan_L1` | Layer1(8) | mean(ctx) | 纯微窗口行为特征 |
| `Bchan_Spatial_L1` | Spatial+L1(10) | mean(ctx) | **V2 BCBest** — 去掉 L2-broadcast 提升 +1.2pp |
| `Bchan_Spatial_L1_L2bc` | Spatial+L1+L2-broadcast(18) | mean(ctx) | **V2 BC-Full** — L2 进 b_in 反而下降（DC 偏置干扰 IIB） |

### Group B：OIB 条件 = `fused(ctx + L2 projection)`

| 变体名 | `b_in` 内容（dim） | OIB 条件 | 设计意图 |
|---|---|---|---|
| `Bchan_Spatial_L2oib` | Spatial(2) | fused(ctx+L2) | **V2 OIBBest** — L2 通过 OIB 门注入 |
| `Bchan_Spatial_L1_L2oib` | Spatial+L1(10) | fused(ctx+L2) | **V2 Hybrid** — BCBest b_in + OIBBest L2门 |
| `Bchan_Full_L2oib` | Spatial+L1+L2bc(18) | fused(ctx+L2) | 新变体 — 双路 L2 是否互补？ |

**V2 探索参考值（非固定 seed）：**
- BCBest (Spatial+L1) ≈ 0.935 F1-Issue
- OIBBest (Spatial, L2-OIB) ≈ 0.926 F1-Issue  
- Hybrid (Spatial+L1, L2-OIB) ≈ 0.929 F1-Issue

消融在完全相同的 **participant-level 5-fold CV** 条件下运行（固定 seed=42、20 轮、取最优 val-F1 checkpoint），
F1 最高的变体写入 `archive/ablation/best_variant.json` 供 `baselines.ipynb` 使用。


In [ ]:
from shared.training import run_cv

# Run (or resume) participant-level 5-fold CV for each variant.
for exp_name, ds_variant in datasets.items():
    print(f"\n{'═'*70}\n▶ Variant: {exp_name}\n{'═'*70}")
    result = run_cv(
        registry[exp_name],
        ds_variant,
        cache_dir=ABL_DIR / exp_name,
        n_folds=5,
        seed=42,
        val_policy="rotate",
        verbose=True,
    )
    results[exp_name] = result

print("\n✅ Ablation complete (participant-level 5-fold CV).")


## §6  5-fold CV `b_in` 消融实验

对 7 个 `b_in` 通道变体分别运行（或续跑）完整的 participant-level 5-fold CV 实验。

- **Group A**（4 个）：固定 `OIB cond = mean(ctx)`，逐步增减 b_in 通道
- **Group B**（3 个）：固定 L2 通过 OIB 门注入，对比 b_in 中是否含 L1/L2-broadcast

结果缓存到 `archive/ablation/<variant>/fold_NN.json`，  
所有 fold 完成后写入 `final_report.json`。  
**重复运行此单元格是安全的** — 已完成的 fold 会直接从缓存加载，跳过训练。


In [ ]:
# ── Run all ChronosXOfficial ablation variants (resumable) ───────────────────
abl_results = run_all_models(
    registry     = CHRONOSX_VARIANTS,   # 7 Official variants
    ds           = ds,
    archive_root = ABL_DIR,
    names        = None,   # None = run all
    verbose      = True,
)
print("\n✅ Ablation complete.")

## §7  Results Visualization

In [ ]:
abl_df = results_to_df(abl_results)
print("── Ablation summary ──────────────────────────────────────────────────────")
display(abl_df)

In [ ]:
plot_f1_leaderboard(
    abl_df,
    title    = "ChronosX Variants — F1-Issue (5-fold CV mean)",
    save_path= str(ABL_DIR / "fig_f1_leaderboard.png"),
)
plot_prf_grouped(
    abl_df,
    top_k    = len(abl_df),
    title    = "ChronosX Variants — Precision / Recall / F1-Issue",
    save_path= str(ABL_DIR / "fig_prf.png"),
)

In [ ]:
# Confusion matrices for all variants
for name in abl_results:
    plot_mean_confmat(abl_results, name)

# Training curves
for name, rep in abl_results.items():
    if rep.get("histories"):
        plot_training_curves(
            rep["histories"],
            title     = f"{name} — fold-averaged training curves",
            save_path = str(ABL_DIR / f"fig_curves_{name}.png"),
        )

## §8  Statistical Tests

For each pair (variant A vs. variant B):
- Normality check (Shapiro-Wilk) → choose paired-t or Wilcoxon
- Bootstrap 95% CI on mean difference
- Effect size (Cohen's d for t-test, Cliff's δ for Wilcoxon)

In [ ]:
# Scott-Knott ESD ranking
sk_df = scott_knott_esd(abl_results, metric="f1_issue")
print("Scott-Knott ESD tiers:")
display(sk_df)

best_variant = sk_df.loc[sk_df["rank_tier"] == 1, "model"].iloc[0]
print(f"\n★  Best ablation variant: {best_variant}")

plot_scott_knott(
    sk_df,
    proposed  = best_variant,
    metric    = "f1_issue",
    title     = "Scott-Knott ESD — ChronosX Variants",
    save_path = str(ABL_DIR / "fig_scott_knott.png"),
)

In [ ]:
# 显著性表：最优变体 vs 其余 6 个变体
sig_df = significance_table(abl_results, proposed=best_variant, metric="f1_issue")
print(f"Significance: {best_variant} vs other Official variants")
display(sig_df)

## §9  最优变体汇总

将消融实验中 F1-Issue 最高的 Official 变体写入 `archive/ablation/best_variant.json`，
`baselines.ipynb` 启动时读取此文件，自动将该变体作为"我们的方法"与 12 条 baseline 进行对比。

In [ ]:
import json as _json

best_summary = {
    "best_variant": best_variant,
    "f1_issue":     float(abl_df.loc[abl_df["model"] == best_variant, "f1_issue"].iloc[0]),
    "f1_macro":     float(abl_df.loc[abl_df["model"] == best_variant, "f1_macro"].iloc[0]),
}
out_path = ABL_DIR / "best_variant.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(_json.dumps(best_summary, indent=2))
print(f"✅ Saved: {out_path}")
print(_json.dumps(best_summary, indent=2))